In [314]:
import pandas as pd
import numpy as np
from math import log2
from math import sqrt
from numpy import asarray
import warnings
warnings.filterwarnings('ignore')


In [315]:
# calculate the kl divergence
def kl_divergence(p, q):
 return sum(p[i] * log2(p[i]/q[i]) for i in range(len(p)))
 
# calculate the js divergence
def js_divergence(p, q):
 m = 0.5 * (p + q)
 return 0.5 * kl_divergence(p, m) + 0.5 * kl_divergence(q, m)

In [316]:
data_1 = pd.read_csv('../data/entropy_data/flan1.csv', index_col=[0])
data_2 = pd.read_csv('../data/entropy_data/flan2.csv', index_col=[0])
data_3 = pd.read_csv('../data/entropy_data/flan3.csv', index_col=[0])
data_3

,Unnamed: 0,uniqueID,trigger_event,label,roleplay,content,zero-shot,few-shot
0,0,poli_ukdata_24-0,political,1,"('not sexist', 0.9166759)","('sexist', 0.9298217)","('sexist', 0.9226115)","('sexist', 0.94947547)"
1,1,poli_ukdata_24-1,political,0,"('not sexist', 0.9407921)","('sexist', 0.9165034)","('sexist', 0.971445)","('sexist', 0.9745518)"
2,2,poli_ukdata_24-2,political,0,"('not sexist', 0.988616)","('not sexist', 0.99873954)","('not sexist', 0.999465)","('not sexist', 0.9994353)"
3,3,poli_ukdata_24-3,political,0,"('not sexist', 0.9613606)","('not sexist', 0.9449113)","('not sexist', 0.969962)","('not sexist', 0.984799)"
4,4,poli_ukdata_24-4,political,0,"('not sexist', 0.97983086)","('not sexist', 0.98537827)","('not sexist', 0.99379426)","('not sexist', 0.99710214)"
...,...,...,...,...,...,...,...,...
1268,1268,poli_ukdata_24-1268,political,0,"('not sexist', 0.99553406)","('not sexist', 0.9910835)","('not sexist', 0.9970583)","('not sexist', 0.99882257)"
1269,1269,poli_ukdata_24-1269,political,0,"('not sexist', 0.9823745)","('not sexist', 0.993674)","('not sexist', 0.9942519)","('not sexist', 0.9948062)"
1270,1270,poli_ukdata_24-1270,political,0,"('not sexist', 0.98172885)","('not sexist', 0.9963051)","('not sexist', 0.99849415)","('not sexist', 0.9986372)"
1271,1271,poli_ukdata_24-1271,political,0,"('not sexist', 0.9883831)","('not sexist', 0.9962468)","('not sexist', 0.998552)","('not sexist', 0.9989954)"


In [317]:
type(tuple(data_1['roleplay'].iloc[0]))

tuple

In [318]:
data_r = data_1[['uniqueID', 'trigger_event', 'label', 'roleplay']]
data_r['roleplay'] = data_r['roleplay'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data_r['pred_1'] = data_r['roleplay'].str[0]
data_r['prob_1'] = data_r['roleplay'].str[1]
data_r = data_r.drop(columns='roleplay')
data_r['prob_1'] = data_r['prob_1'].astype('str').apply(lambda x: float(x.strip()))

data_r = pd.merge(data_r, data_2[['uniqueID', 'roleplay']], on = 'uniqueID', how='left')
data_r['roleplay'] = data_r['roleplay'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data_r['pred_2'] = data_r['roleplay'].str[0]
data_r['prob_2'] = data_r['roleplay'].str[1]
data_r['prob_2'] = data_r['prob_2'].astype('str').apply(lambda x: float(x.strip()))
data_r = data_r.drop(columns=['roleplay'])

data_r = pd.merge(data_r, data_3[['uniqueID', 'roleplay']], on = 'uniqueID', how='left')
data_r['roleplay'] = data_r['roleplay'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data_r['pred_3'] = data_r['roleplay'].str[0]
data_r['prob_3'] = data_r['roleplay'].str[1]
data_r['prob_3'] = data_r['prob_3'].astype('str').apply(lambda x: float(x.strip()))
data_r = data_r.drop(columns=['roleplay'])

## drop nan values as they are not present as much (n=1, in our case) and do not give us the right information
## the original prediction in those instances indicate that the model had taken to their default value instead of predicting a label/
data_r = data_r.dropna(subset=['prob_1', 'prob_2', 'prob_3'], how = 'all').reset_index(drop=True)

data_r

,uniqueID,trigger_event,label,pred_1,prob_1,pred_2,prob_2,pred_3,prob_3
0,poli_ukdata_24-0,political,1,'not sexist',0.916676,'not sexist',0.916676,'not sexist',0.916676
1,poli_ukdata_24-1,political,0,'not sexist',0.940792,'not sexist',0.940792,'not sexist',0.940792
2,poli_ukdata_24-2,political,0,'not sexist',0.988616,'not sexist',0.988616,'not sexist',0.988616
3,poli_ukdata_24-3,political,0,'not sexist',0.961361,'not sexist',0.961361,'not sexist',0.961361
4,poli_ukdata_24-4,political,0,'not sexist',0.979831,'not sexist',0.979831,'not sexist',0.979831
...,...,...,...,...,...,...,...,...,...
1267,poli_ukdata_24-1268,political,0,'not sexist',0.995534,'not sexist',0.995534,'not sexist',0.995534
1268,poli_ukdata_24-1269,political,0,'not sexist',0.982375,'not sexist',0.982375,'not sexist',0.982375
1269,poli_ukdata_24-1270,political,0,'not sexist',0.981729,'not sexist',0.981729,'not sexist',0.981729
1270,poli_ukdata_24-1271,political,0,'not sexist',0.988383,'not sexist',0.988383,'not sexist',0.988383


In [319]:
data_r['full agreement'] = data_r.apply(lambda row: row['pred_1'] == row['pred_2'] == row['pred_3'], axis=1)
data_r['full agreement'].value_counts()

full agreement
True    1272
Name: count, dtype: int64

In [320]:
# define distributions
p_r = asarray(data_r['prob_1'].to_list())
q_r = asarray(data_r['prob_3'].to_list())
# calculate JS(P || Q)
js_pq_r = js_divergence(p_r, q_r)
1- js_pq_r

0.9999999999968094

In [321]:
# define distributions
p_r = asarray(data_r['prob_1'].to_list())
q_r = asarray(data_r['prob_2'].to_list())
# calculate JS(P || Q)
js_pq_r = js_divergence(p_r, q_r)
1- js_pq_r

0.9999999999968094

In [322]:
# define distributions
p_r = asarray(data_r['prob_2'].to_list())
q_r = asarray(data_r['prob_3'].to_list())
# calculate JS(P || Q)
js_pq_r = js_divergence(p_r, q_r)
1- js_pq_r

1.0

In [323]:
np.mean([0.9999999999968094, 0.9999999999968094, 1.0]), np.std([0.9999999999968094, 0.9999999999968094, 1.0])

(0.9999999999978729, 1.504043903043767e-12)

In [ ]:
data_c = data_1[['uniqueID', 'trigger_event', 'label', 'content']]
data_c['content'] = data_c['content'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data_c['pred_1'] = data_c['content'].str[0]
data_c['prob_1'] = data_c['content'].str[1]
data_c = data_c.drop(columns='content')
data_c['prob_1'] = data_c['prob_1'].astype('str').apply(lambda x: float(x.strip()))

data_c = pd.merge(data_c, data_2[['uniqueID', 'content']], on = 'uniqueID', how='left')
data_c['content'] = data_c['content'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data_c['pred_2'] = data_c['content'].str[0]
data_c['prob_2'] = data_c['content'].str[1]
data_c['prob_2'] = data_c['prob_2'].astype('str').apply(lambda x: float(x.strip()))
data_c = data_c.drop(columns=['content'])

data_c = pd.merge(data_c, data_3[['uniqueID', 'content']], on = 'uniqueID', how='left')
data_c['content'] = data_c['content'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data_c['pred_3'] = data_c['content'].str[0]
data_c['prob_3'] = data_c['content'].str[1]
data_c['prob_3'] = data_c['prob_3'].astype('str').apply(lambda x: float(x.strip()))
data_c = data_c.drop(columns=['content'])

## drop nan values as they are not present as much (n=1, in our case) and do not give us the right information
## the original prediction in those instances indicate that the model had taken to their default value instead of predicting a label/
data_c = data_c.dropna(subset=['prob_1', 'prob_2', 'prob_3'], how = 'all').reset_index(drop=True)

data_c

,uniqueID,trigger_event,label,pred_1,prob_1,pred_2,prob_2,pred_3,prob_3
0,poli_ukdata_24-0,political,1,'sexist',0.929822,'sexist',0.929822,'sexist',0.929822
1,poli_ukdata_24-1,political,0,'sexist',0.916504,'sexist',0.916503,'sexist',0.916503
2,poli_ukdata_24-2,political,0,'not sexist',0.998740,'not sexist',0.998740,'not sexist',0.998740
3,poli_ukdata_24-3,political,0,'not sexist',0.944911,'not sexist',0.944911,'not sexist',0.944911
4,poli_ukdata_24-4,political,0,'not sexist',0.985378,'not sexist',0.985378,'not sexist',0.985378
...,...,...,...,...,...,...,...,...,...
1267,poli_ukdata_24-1268,political,0,'not sexist',0.991084,'not sexist',0.991084,'not sexist',0.991084
1268,poli_ukdata_24-1269,political,0,'not sexist',0.993674,'not sexist',0.993674,'not sexist',0.993674
1269,poli_ukdata_24-1270,political,0,'not sexist',0.996305,'not sexist',0.996305,'not sexist',0.996305
1270,poli_ukdata_24-1271,political,0,'not sexist',0.996247,'not sexist',0.996247,'not sexist',0.996247


In [324]:
data_c['full agreement'] = data_c.apply(lambda row: row['pred_1'] == row['pred_2'] == row['pred_3'], axis=1)
data_c['full agreement'].value_counts()

full agreement
True    1272
Name: count, dtype: int64

In [325]:
# define distributions
p_c = asarray(data_c['prob_1'].to_list())
q_c = asarray(data_c['prob_3'].to_list())
# calculate JS(P || Q)
js_pq_c = js_divergence(p_c, q_c)
1- js_pq_c

0.9999999999976137

In [326]:
# define distributions
p_c = asarray(data_c['prob_2'].to_list())
q_c = asarray(data_c['prob_3'].to_list())
# calculate JS(P || Q)
js_pq_c = js_divergence(p_c, q_c)
1- js_pq_c

1.0

In [327]:
# define distributions
p_c = asarray(data_c['prob_1'].to_list())
q_c = asarray(data_c['prob_2'].to_list())
# calculate JS(P || Q)
js_pq_c = js_divergence(p_c, q_c)
1- js_pq_c

0.9999999999976137

In [328]:
np.mean([0.9999999999976137, 0.9999999999976137, 1.0]), np.std([0.9999999999976137, 0.9999999999976137, 1.0])

(0.9999999999984092, 1.1249189108403681e-12)

In [329]:
data_z = data_1[['uniqueID', 'trigger_event', 'label', 'zero-shot']]
data_z['zero-shot'] = data_z['zero-shot'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data_z['pred_1'] = data_z['zero-shot'].str[0]
data_z['prob_1'] = data_z['zero-shot'].str[1]
data_z = data_z.drop(columns='zero-shot')
data_z['prob_1'] = data_z['prob_1'].astype('str').apply(lambda x: float(x.strip()))

data_z = pd.merge(data_z, data_2[['uniqueID', 'zero-shot']], on = 'uniqueID', how='left')
data_z['zero-shot'] = data_z['zero-shot'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data_z['pred_2'] = data_z['zero-shot'].str[0]
data_z['prob_2'] = data_z['zero-shot'].str[1]
data_z['prob_2'] = data_z['prob_2'].astype('str').apply(lambda x: float(x.strip()))
data_z = data_z.drop(columns=['zero-shot'])

data_z = pd.merge(data_z, data_3[['uniqueID', 'zero-shot']], on = 'uniqueID', how='left')
data_z['zero-shot'] = data_z['zero-shot'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data_z['pred_3'] = data_z['zero-shot'].str[0]
data_z['prob_3'] = data_z['zero-shot'].str[1]
data_z['prob_3'] = data_z['prob_3'].astype('str').apply(lambda x: float(x.strip()))
data_z = data_z.drop(columns=['zero-shot'])

## drop nan values as they are not present as much (n=1, in our case) and do not give us the right information
## the original prediction in those instances indicate that the model had taken to their default value instead of predicting a label/
data_z = data_z.dropna(subset=['prob_1', 'prob_2', 'prob_3'], how = 'all').reset_index(drop=True)

data_z

,uniqueID,trigger_event,label,pred_1,prob_1,pred_2,prob_2,pred_3,prob_3
0,poli_ukdata_24-0,political,1,'sexist',0.922612,'sexist',0.922612,'sexist',0.922612
1,poli_ukdata_24-1,political,0,'sexist',0.971445,'sexist',0.971445,'sexist',0.971445
2,poli_ukdata_24-2,political,0,'not sexist',0.999465,'not sexist',0.999465,'not sexist',0.999465
3,poli_ukdata_24-3,political,0,'not sexist',0.969962,'not sexist',0.969962,'not sexist',0.969962
4,poli_ukdata_24-4,political,0,'not sexist',0.993794,'not sexist',0.993794,'not sexist',0.993794
...,...,...,...,...,...,...,...,...,...
1267,poli_ukdata_24-1268,political,0,'not sexist',0.997058,'not sexist',0.997058,'not sexist',0.997058
1268,poli_ukdata_24-1269,political,0,'not sexist',0.994252,'not sexist',0.994252,'not sexist',0.994252
1269,poli_ukdata_24-1270,political,0,'not sexist',0.998494,'not sexist',0.998494,'not sexist',0.998494
1270,poli_ukdata_24-1271,political,0,'not sexist',0.998552,'not sexist',0.998552,'not sexist',0.998552


In [330]:
data_z['full agreement'] = data_z.apply(lambda row: row['pred_1'] == row['pred_2'] == row['pred_3'], axis=1)
data_z['full agreement'].value_counts()

full agreement
True    1272
Name: count, dtype: int64

In [331]:
# define distributions
p_z = asarray(data_z['prob_1'].to_list())
q_z = asarray(data_z['prob_3'].to_list())
# calculate JS(P || Q)
js_pq_z = js_divergence(p_z, q_z)
1- js_pq_z

0.999999999998238

In [332]:
# define distributions
p_z = asarray(data_z['prob_1'].to_list())
q_z = asarray(data_z['prob_2'].to_list())
# calculate JS(P || Q)
js_pq_z = js_divergence(p_z, q_z)
1- js_pq_z

0.999999999998238

In [333]:
# define distributions
p_z = asarray(data_z['prob_2'].to_list())
q_z = asarray(data_z['prob_3'].to_list())
# calculate JS(P || Q)
js_pq_z = js_divergence(p_z, q_z)
1- js_pq_z

1.0

In [334]:
np.mean([0.999999999998238, 0.999999999998238, 1.0]), np.std([0.999999999998238, 0.999999999998238, 1.0])

(0.9999999999988253, 8.306312478834097e-13)

In [335]:
data_f = data_1[['uniqueID', 'trigger_event', 'label', 'few-shot']]
data_f['few-shot'] = data_f['few-shot'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data_f['pred_1'] = data_f['few-shot'].str[0]
data_f['prob_1'] = data_f['few-shot'].str[1]
data_f = data_f.drop(columns='few-shot')
data_f['prob_1'] = data_f['prob_1'].astype('str').apply(lambda x: float(x.strip()))

data_f = pd.merge(data_f, data_2[['uniqueID', 'few-shot']], on = 'uniqueID', how='left')
data_f['few-shot'] = data_f['few-shot'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data_f['pred_2'] = data_f['few-shot'].str[0]
data_f['prob_2'] = data_f['few-shot'].str[1]
data_f['prob_2'] = data_f['prob_2'].astype('str').apply(lambda x: float(x.strip()))
data_f = data_f.drop(columns=['few-shot'])

data_f = pd.merge(data_f, data_3[['uniqueID', 'few-shot']], on = 'uniqueID', how='left')
data_f['few-shot'] = data_f['few-shot'].apply(lambda x: tuple(x.replace('(','').replace(')','').split(',')))
data_f['pred_3'] = data_f['few-shot'].str[0]
data_f['prob_3'] = data_f['few-shot'].str[1]
data_f['prob_3'] = data_f['prob_3'].astype('str').apply(lambda x: float(x.strip()))
data_f = data_f.drop(columns=['few-shot'])

## drop nan values as they are not present as much (n=1, in our case) and do not give us the right information
## the original prediction in those instances indicate that the model had taken to their default value instead of predicting a label/
data_f = data_f.dropna(subset=['prob_1', 'prob_2', 'prob_3'], how = 'all').reset_index(drop=True)

data_f

,uniqueID,trigger_event,label,pred_1,prob_1,pred_2,prob_2,pred_3,prob_3
0,poli_ukdata_24-0,political,1,'sexist',0.949475,'sexist',0.949475,'sexist',0.949475
1,poli_ukdata_24-1,political,0,'sexist',0.974552,'sexist',0.974552,'sexist',0.974552
2,poli_ukdata_24-2,political,0,'not sexist',0.999435,'not sexist',0.999435,'not sexist',0.999435
3,poli_ukdata_24-3,political,0,'not sexist',0.984799,'not sexist',0.984799,'not sexist',0.984799
4,poli_ukdata_24-4,political,0,'not sexist',0.997102,'not sexist',0.997102,'not sexist',0.997102
...,...,...,...,...,...,...,...,...,...
1267,poli_ukdata_24-1268,political,0,'not sexist',0.998823,'not sexist',0.998823,'not sexist',0.998823
1268,poli_ukdata_24-1269,political,0,'not sexist',0.994806,'not sexist',0.994806,'not sexist',0.994806
1269,poli_ukdata_24-1270,political,0,'not sexist',0.998637,'not sexist',0.998637,'not sexist',0.998637
1270,poli_ukdata_24-1271,political,0,'not sexist',0.998995,'not sexist',0.998995,'not sexist',0.998995


In [336]:
data_f['full agreement'] = data_f.apply(lambda row: row['pred_1'] == row['pred_2'] == row['pred_3'], axis=1)
data_f['full agreement'].value_counts()

full agreement
True    1272
Name: count, dtype: int64

In [337]:
# define distributions
p_f = asarray(data_f['prob_1'].to_list())
q_f = asarray(data_f['prob_3'].to_list())
# calculate JS(P || Q)
js_pq_f = js_divergence(p_f, q_f)
1- js_pq_f

0.9999999999985733

In [297]:
# define distributions
p_f = asarray(data_f['prob_1'].to_list())
q_f = asarray(data_f['prob_2'].to_list())
# calculate JS(P || Q)
js_pq_f = js_divergence(p_f, q_f)
1- js_pq_f

0.9999999999985733

In [298]:
# define distributions
p_f = asarray(data_f['prob_2'].to_list())
q_f = asarray(data_f['prob_3'].to_list())
# calculate JS(P || Q)
js_pq_f = js_divergence(p_f, q_f)
1- js_pq_f

1.0

In [300]:
np.mean([0.9999999999985733, 0.9999999999985733, 1.0]), np.std([0.9999999999985733, 0.9999999999985733, 1.0])

(0.9999999999990488, 6.725752769573956e-13)